# Stock Trend Classifier
Predicts whether a stock is **trending up** over the next N days using common technical indicators.

**Model:** Random Forest (lightweight, CPU-only, trains in seconds)

**Features:** RSI, MACD, SMA crossover, Bollinger Band position, volume change, momentum

**Label:** 1 if price N days later is > X% above current price, else 0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

np.random.seed(42)
print('All imports OK')

## 1. Data
Synthetic OHLCV data via Geometric Brownian Motion — swap the cell below with `yfinance` or a CSV if you have real data.

In [ ]:
def generate_ohlcv(n_days: int = 1500, mu: float = 0.0003, sigma: float = 0.015, start_price: float = 100.0) -> pd.DataFrame:
    """Geometric Brownian Motion price series with synthetic volume."""
    returns = np.random.normal(mu, sigma, n_days)
    close = start_price * np.cumprod(1 + returns)

    noise = lambda scale: np.abs(np.random.normal(0, scale, n_days))
    high  = close * (1 + noise(0.007))
    low   = close * (1 - noise(0.007))
    open_ = np.roll(close, 1);  open_[0] = start_price

    base_vol = 1_000_000
    volume = base_vol + base_vol * 0.5 * np.random.randn(n_days)
    volume = np.clip(volume, 100_000, None).astype(int)

    dates = pd.date_range(end='2026-04-20', periods=n_days, freq='B')
    return pd.DataFrame({'open': open_, 'high': high, 'low': low, 'close': close, 'volume': volume}, index=dates)


# --- Swap this block for real data ---
# import yfinance as yf
# df = yf.download('AAPL', start='2018-01-01', end='2026-01-01')
# df.columns = df.columns.get_level_values(0).str.lower()
# --------------------------------------

df = generate_ohlcv(n_days=1500)
print(df.tail())
df['close'].plot(title='Simulated Close Price', figsize=(12, 3))
plt.tight_layout()
plt.show()

## 2. Feature Engineering

In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def compute_macd(series: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9):
    ema_fast   = series.ewm(span=fast,   adjust=False).mean()
    ema_slow   = series.ewm(span=slow,   adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    c = df['close'].copy()
    v = df['volume'].copy()

    # --- Trend / moving averages ---
    df['sma20']       = c.rolling(20).mean()
    df['sma50']       = c.rolling(50).mean()
    df['sma_cross']   = df['sma20'] - df['sma50']          # positive = bullish cross
    df['price_sma20'] = c / df['sma20'] - 1                # distance from SMA20

    # --- Momentum ---
    df['mom5']   = c.pct_change(5)
    df['mom10']  = c.pct_change(10)
    df['mom20']  = c.pct_change(20)

    # --- RSI ---
    df['rsi14'] = compute_rsi(c, 14)

    # --- MACD ---
    df['macd'], df['macd_signal'] = compute_macd(c)
    df['macd_hist'] = df['macd'] - df['macd_signal']       # histogram: direction strength

    # --- Bollinger Bands ---
    roll20        = c.rolling(20)
    bb_mid        = roll20.mean()
    bb_std        = roll20.std()
    df['bb_pct']  = (c - (bb_mid - 2 * bb_std)) / (4 * bb_std)  # 0=lower band, 1=upper band

    # --- Volatility ---
    df['atr14']   = (df['high'] - df['low']).rolling(14).mean() / c  # normalised ATR

    # --- Volume ---
    df['vol_chg']  = v.pct_change()
    df['vol_ratio'] = v / v.rolling(20).mean()             # current vs average volume

    return df


df = add_features(df)
print('Features added:', df.shape)

## 3. Label: Trending Up?
Label = 1 if the close price **5 trading days from now** is at least **1%** higher than today.

In [ ]:
HORIZON   = 5     # days ahead to predict
THRESHOLD = 0.01  # minimum % gain to call it "trending up"

df['future_return'] = df['close'].shift(-HORIZON) / df['close'] - 1
df['label']         = (df['future_return'] > THRESHOLD).astype(int)

print('Label distribution:')
print(df['label'].value_counts())
print(f'\nUp days: {df["label"].mean():.1%}')

## 4. Train / Test Split

In [ ]:
FEATURE_COLS = [
    'sma_cross', 'price_sma20',
    'mom5', 'mom10', 'mom20',
    'rsi14',
    'macd', 'macd_signal', 'macd_hist',
    'bb_pct',
    'atr14',
    'vol_chg', 'vol_ratio',
]

data = df[FEATURE_COLS + ['label']].dropna()
X = data[FEATURE_COLS].values
y = data['label'].values

# Time-series split: first 80% train, last 20% test (no shuffling — avoids lookahead)
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

## 5. Train Random Forest

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,          # shallow trees reduce overfitting on noisy price data
    min_samples_leaf=20,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)

model.fit(X_train, y_train)
print('Training complete')

## 6. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Down/Flat', 'Up']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Down/Flat', 'Up'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('Confusion Matrix')

# Predicted probability distribution
axes[1].hist(y_prob[y_test == 0], bins=30, alpha=0.6, label='Actual Down/Flat', color='tomato')
axes[1].hist(y_prob[y_test == 1], bins=30, alpha=0.6, label='Actual Up',        color='steelblue')
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted P(Up)')
axes[1].set_title('Prediction Score Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importances (Mean Decrease in Impurity)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Predict on New Data
Pass in the latest row of indicators to get a trend prediction.

In [ ]:
def predict_trend(df_with_features: pd.DataFrame, model, feature_cols: list, n_rows: int = 1) -> pd.DataFrame:
    """Return trend predictions for the last n_rows of df."""
    latest = df_with_features[feature_cols].dropna().tail(n_rows)
    probs  = model.predict_proba(latest.values)[:, 1]
    preds  = (probs >= 0.5).astype(int)
    return pd.DataFrame({
        'date':       latest.index,
        'p_up':       probs.round(3),
        'prediction': np.where(preds == 1, 'UP', 'DOWN/FLAT'),
    })


result = predict_trend(df, model, FEATURE_COLS, n_rows=3)
print(result.to_string(index=False))